##### This notebook's goal:
To preprocess, merge, and otherwise prepare the original datasets (JSON) while converting them to CSVs. This includes some text normalisation, tokenisation, and feature engineering steps.

##### Author(s):
- (Ahn) Michael Howell - Human Language Technology Masters - Sociolinguist & Agentic AI Engineer - ahn@equita-tech.com

#### Gather imports

In [38]:
import numpy as np
import pandas as pd
import re # For text processing, pattern recognition
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS # For focusing NLP analysis on contentful words

#### Load original data

In [39]:
## LOAD DATA
reddit_url = "https://raw.githubusercontent.com/taivop/joke-dataset/refs/heads/master/reddit_jokes.json"
stupid_stuff_url = "https://raw.githubusercontent.com/taivop/joke-dataset/refs/heads/master/stupidstuff.json"
wocka_url = "https://raw.githubusercontent.com/taivop/joke-dataset/refs/heads/master/wocka.json"

reddit_df = pd.read_json(reddit_url) # Extra columns: title, score
ss_df = pd.read_json(stupid_stuff_url) # Extra columns: rating, category
wocka_df = pd.read_json(wocka_url) # Extra columns: title, category

##### Preprocessing: tidy initial columns & merging

In [40]:
# Add site column to each
reddit_df["website"] = "reddit"
ss_df["website"] = "stupidstuff"
wocka_df["website"] = "wocka"

# Ensure each dataset has all necessary columns
for d in (reddit_df, ss_df, wocka_df):
    for col in ['title', 'category', 'score', 'rating']:
        if col not in d.columns:
            d[col] = pd.NA

## MERGE DATASETS
merged_df = pd.concat([reddit_df, ss_df, wocka_df], ignore_index=True)

# Ensure text columns are real strings (empty if missing)
merged_df['title'] = merged_df['title'].fillna('')
merged_df['body'] = merged_df['body'].fillna('')
merged_df['category'] = merged_df['category'].fillna('')

# Ensure score is numeric
merged_df['score'] = pd.to_numeric(merged_df['score'], errors='coerce')

## REVIEW UPDATES
def show_df_details(details_to_include = []):
    if "head" in details_to_include:
        print(f"\n** merged_df.head:\n{merged_df.head()}\n")
    if "tail" in details_to_include:
        print(f"\n** merged_df.tail:\n{merged_df.tail()}\n")
    if "info" in details_to_include:
        merged_df.info()
        # print(f"\n** merged_df.info:\n{merged_df.info(verbose = 'true')}\n")
    if "describe" in details_to_include:
        print(f"\n** merged_df.describe:\n{merged_df.describe()}\n")
    if "describe-all" in details_to_include:
        print(f"\n** merged_df.describe:\n{merged_df.describe(include = 'all')}\n")

show_df_details(["head", "tail", "info", "describe-all"])
# Consider if rating & score can be immediately merged > no, they are not on the same scale
# Rating is 0 to 5, while score is upvotes count; TODO: reflect on binning strategy
# to fit into 5-part scale: VERYLOW, LOW, MID, HIGH, VERYHIGH, as user_rating column
# TODO: look at distribution of score column, review outliers
# TODO: reflect on if the 5-part binning makes more sense or keeping numerics; maybe both
# Score is extremely skewed (median = 3, max = 48,526, 75% quantile = 16, means 75% of jokes have 16 upvotes or less)
# Standard deviation = 936; variance is mostly caused by viral jokes > should use quantiles to bin
print(f"\nscore & rating .describe:\n{merged_df[['score','rating']].describe(include='all')}\n")
# TODO: check on if the IDs will be a challenge > it may make most sense to just keep reddit data only

## FILTER - to simplify data intake (We decided to keep only Reddit data)
websites_to_keep = ["reddit"]
columns_to_keep = ["body", "id", "title", "score", "website"]
merged_df = merged_df.loc[merged_df['website'].isin(websites_to_keep), columns_to_keep].copy()
print(f"\n** AFTER FILTER: merged_df.head:\n{merged_df.head()}\n")

/tmp/ipykernel_63002/1150353287.py:13: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  merged_df = pd.concat([reddit_df, ss_df, wocka_df], ignore_index=True)



** merged_df.head:
                                                body      id  score  \
0  Now I have to say "Leroy can you please paint ...  5tz52q    1.0   
1  Pizza doesn't scream when you put it in the ov...  5tz4dd    0.0   
2  ...and being there really helped me learn abou...  5tz319    0.0   
3  A Sunday school teacher is concerned that his ...  5tz2wj    1.0   
4  He got caught trying to sell the two books to ...  5tz1pc    0.0   

                                               title website category  rating  
0   I hate how you cant even say black paint anymore  reddit              NaN  
1  What's the difference between a Jew in Nazi Ge...  reddit              NaN  
2                     I recently went to America....  reddit              NaN  
3  Brian raises his hand and says, “He’s in Heaven.”  reddit              NaN  
4  You hear about the University book store worke...  reddit              NaN  


** merged_df.tail:
                                                    

##### Preprocessing: gather whole text & initial text features

In [41]:
# Add feature: whole_text (string) - the first and second parts of the jokes combined
merged_df['whole_text'] = (merged_df['title'] + ' ' + merged_df['body']).str.strip()

# Add feature: whole_text_normalised (string) - to support further NLP analysis
def normalise_text(s):
    s = s.lower() # make lowercase
    s = re.sub(r"[^a-z0-9\s]", " ", s) # remove non-text characters
    s = re.sub(r"\s+", " ", s)  # collapse multiple spaces to a single space
    s = s.strip() # remove whitespace from ends
    return s
merged_df['whole_text_normalised'] = merged_df['whole_text'].apply(normalise_text)

# Add feature: whole_text_normalised_tokens (list) - to support further NLP analysis
merged_df['whole_text_normalised_tokens'] = merged_df['whole_text_normalised'].str.split()

# Add feature: overall_length (int) - count of the words in the text (not normalised, stop words remain)
merged_df['overall_length'] = merged_df['whole_text_normalised_tokens'].apply(len)

# Add feature: distinct_words (list) - the list of different words in the text (stop words removed)
CONTRACTED_STOP_WORDS = {"im","ive","youre","dont","didnt","hes","shes","theyre","weve","cant","wont","thats"}
STOP_WORDS_ALL = set(ENGLISH_STOP_WORDS) | CONTRACTED_STOP_WORDS
merged_df['distinct_words'] = merged_df['whole_text_normalised_tokens'].apply(
    lambda toks: list(set(t for t in toks if t not in STOP_WORDS_ALL))
)

# Add feature: distinct_words_count (int) - the number of different words used
merged_df['distinct_words_count'] = merged_df['distinct_words'].apply(len)

## REVIEW UPDATES
show_df_details(["head", "tail", "info", "describe"])


** merged_df.head:
                                                body      id  \
0  Now I have to say "Leroy can you please paint ...  5tz52q   
1  Pizza doesn't scream when you put it in the ov...  5tz4dd   
2  ...and being there really helped me learn abou...  5tz319   
3  A Sunday school teacher is concerned that his ...  5tz2wj   
4  He got caught trying to sell the two books to ...  5tz1pc   

                                               title  score website  \
0   I hate how you cant even say black paint anymore    1.0  reddit   
1  What's the difference between a Jew in Nazi Ge...    0.0  reddit   
2                     I recently went to America....    0.0  reddit   
3  Brian raises his hand and says, “He’s in Heaven.”    1.0  reddit   
4  You hear about the University book store worke...    0.0  reddit   

                                          whole_text  \
0  I hate how you cant even say black paint anymo...   
1  What's the difference between a Jew in Nazi Ge...   


##### Preprocessing: gather theme counts from features

In [42]:
## LIST OUR THEME WORDS

theme_words_gender = [
    "gender","woman","women","girl","girls","lady","ladies","female","females",
    "wife","wives","mother","mothers","mom","moms","aunt","aunts","daughter","daughters","sister","sisters",
    "man","men","boy","boys","gentleman","gentlemen","male","males","cis","cishetero"
    "husband","husbands","father","fathers","dad","dads","uncle","uncles","son","sons","brother","brothers",
    "trans","transgender","transition","mtf","ftm","nonbinary","genderqueer","agender","genderfluid",
    "feminine","masculine","masc","fem","femme","butch","misogyny","misogynist","patriarchy","matriarchy","sexism","sexist",
    "girlfriend","boyfriend","bride","brides","groom","grooms","fiance","fiancee",
    "pregnant","sexist","misogynist","emasculated","motherly","fatherly","girly","manly",
    # Additional from data
    "guy","chick","babe","period","periods",
]

theme_words_ethnicity = [
    "race","black","white","asian","african","afro","caucasian","european","latino","latina","latinx",
    "hispanic","mexican","chinese","japanese","korean","vietnamese","filipino","indian","pakistani","bangladeshi",
    "eastern","arab","persian","turkish","native","indigenous","aboriginal","nations", "western"
    "ethiopian","somali","nigerian","kenyan","congolese","egyptian","moroccan",
    "racist","racism","stereotype","discriminate","discrimination","prejudice","xenophobia","xenophobic",
    "slur","ethnic","ethnicity","skin","melanin","tribe","tribal","minority","majority","ghetto","thug"
    "racist","prejudiced","xenophobic","discriminatory","tribal",
    "segregated","integrated","oppressed","marginalized","marginalised","colonized","colonised",
    # Additional from data - including some nationality and language-group terms
    "jew","latin","nazi","america","american","usa","europe","european","french","france","africa","asia","canada","australia","australian","hawaiian",
    "uk","brit","gb","britain","british","germany","china","mexico","brazil","sweden","israel","israeli","palestine","palestinian","gaza","ireland",
    "russia","russian","ukraine","arab","uae","saudi","arabic","french","english","spanish","portuguese","japanese","thai","irish",
    "kkk","nigger","niggers","chink","chinks","wetback","ethiopian","caucasian","dea","culture","cultural","minorities","jewish"

]

theme_words_sexuality = [
    "sexuality","orientation","gay","gays","homosexual","homo","lesbian","lesbians","bi","bisexual","pansexual","queer",
    "straight","hetero","heterosexual","closet","closeted","coming","outing",
    "lgbt","lgbtq","lgbtqia","pride","rainbow","ally","allies",
    "sex","sexual","sexy","desire","arousal","aroused","intimacy","intimate","lust","flirt","flirting",
    "kink","kinky","fetish","fetishes","hookup","hooking","affair","affairs","cheat","cheating",
    "romance","romantic","date","dating","partner","partners","spouse","spouses","relationship","relationships",
    "wife","wives","husband","husbands","girlfriend","girlfriends","boyfriend","boyfriends",
    "bride","brides","groom","grooms","fiance","fiancee","engaged","engagement",
    "marriage","married","wedding","couple","couples","lover","lovers","crush","crushes",
    "mistress","mistresses","significant","poly","polyamorous","polyamory","monogamous","monogamy",
    "affectionate","lustful","lusted","lusting","promiscuous","flirtatious","desirable","attractive","attracted","attract"
    "intimate","faithful","unfaithful","monogamous","polyamorous",
    # Additional from data
    "daddy","dilf","milf","porn","porno","hooker","aids","std","sti","molestor","pervert","offender","virgin",
    "headjob","handjob","bj","oral","anal","cock","cocks","dick","dicks","clit","clits","cunt","cunts","suck","sucked","fuck","fucked","fucks",
    "viagra","hardon","jerk","jerks","jerked","fucking",
]

theme_words_ability = [
    "ability","disabled","disability","handicap","handicapped","wheelchair","paraplegic","quadriplegic","blind","deaf",
    "mute","impaired","learning","dyslexia","dyslexic","autism","autistic","adhd","audhd","neurodivergent","deformed",
    "weak","weaker","weakness","stupid","stupidity","dumb","dumber","slow","slowness","retard","retarded","special"
    "cripple","crippled","amputee","limp","limping","spelling","misspell","misspelling","typo","grammar","illiterate",
    "brain","damage","mental","illness","health","psychiatric","psychotic","psychosis","depression","anxiety",
    # Additional from data
    "idiot","amnesia","dementia","stutter","impediment","therapy","phonics","cripple","crippled",
    "paralyzed","dumbass","dummy","iq","intelligent","smart","smarter","clever","able"
]

theme_words_age = [
    "age","child","children","kid","kids","toddler","toddlers","baby","babies","infant","infants",
    "teen","teens","teenager","teenagers","youth","youths","minor","minors","adolescent","adolescents",
    "young","younger","youthful","freshman","sophomore","junior","senior",
    "adult","adults","grownup","grownups","middle aged","middleaged","elder","elders","elderly","old","older","aging","aged","senior",
    "grandma","grandmas","grandmother","grandmothers","granny","grannies",
    "grandpa","grandpas","grandfather","grandfathers","granddad","granddads","pensioner","retiree","retirees",
]

theme_words_weight = [
    "weight","fat","fatter","fattest","obese","obesity","overweight","underweight","bmi","massive","giant","huge","chubby","chunky","plump","heavy","heavier","heaviest",
    "big boned","bigboned","plussize","plussize","figured","fullfigured","thick","thicc","rotund","portly",
    "skinny","skinnier","skinniest","thin","thinner","thinnest","slim","slimmer","slimmest","underweight",
    "gaunt","emaciated","anorexic","anorexia","bulimic","bulimia","malnourished","malnutrition",
    # Additional from data
    "bulimics","diet","dieting"
]

theme_words_appearance = [
    "appearance","beauty","ugly","uglier","ugliest","hideous","unattractive","plain","homely","unsightly","grotesque","disfigured","scarred","scar","scarface","wrinkled","wrinkle","wrinkles",
    "beautiful","pretty","prettier","prettiest","gorgeous","handsome","cute","attractive","attracted","unattractive","looking","goodlooking","fine","hot","foxy",
    "blond","blonde","brunette","redhead","ginger","bald","balding","baldness",
    "pimple","pimples","acne","zit","zits","blemish","blemishes","scar","scars",
    "fat","fatter","fattest","obese","chubby","skinny","slim","thin","muscular","fit","ripped","toned",
    # Additional from data
    "hideous",
]

theme_words_class = [
    "class","poverty","poor","poorer","poorest","income","lowincome","lowerclass","working","worker","workingclass","destitute","impoverished","needy","broke",
    "homeless","unhoused","vagrant","beggar","beggars","panhandler","panhandlers","squatter","squatters","ghetto","middleclass","uppermiddleclass","wealthy","rich",
    "richer","richest","affluent","prosperous","opulent","luxurious","privileged","upperclass","aristocrat","aristocracy","noble","nobles","nobility","elite","elites","millionaire","millionaires","billionaire","billionaires",
    "welfare","unemployed","jobless","benefits","subsidized","housing","socialhousing","work","pay","job","afford","affordable","wealth","rundown","struggle","struggling",
    # Additional from data
    "stealing","stolen","theft","robbery","crime","thief","rob","robbed","market","livelihood","wage","wages","salary",
    "house","apartment","rent","buy","purchase","own","owner","homeowner","landlord","boss","employer","supervisor",


]

THEMES = {
    "gender": {"words": theme_words_gender},
    "ethnicity": {"words": theme_words_ethnicity},
    "sexuality": {"words": theme_words_sexuality},
    "ability": {"words": theme_words_ability},
    "age": {"words": theme_words_age},
    "weight": {"words": theme_words_weight},
    "appearance": {"words": theme_words_appearance},
    "class": {"words": theme_words_class}
}

In [43]:
## ADD THEME FEATURES
for theme_name, theme_data in THEMES.items():

    print(f"\n** PROCESSING THEME: {theme_name}\n")

    # Add count feature - how many instances of themed words appear in the joke
    feature_count_name = f"theme_{theme_name}_count"
    merged_df[feature_count_name] = merged_df["whole_text_normalised_tokens"].apply(
        lambda toks: sum(1 for t in toks if t in THEMES[theme_name]["words"])
    )

    print(f"\t> Completed count feature ({theme_name})")

    # Add boolean feature - whether any instances of the themes words appear in the joke
    feature_bool_name = f"theme_{theme_name}_bool"
    merged_df[feature_bool_name] = merged_df["whole_text_normalised_tokens"].apply(
        lambda toks: any(t in THEMES[theme_name]["words"] for t in toks)
    )

    print(f"\t> Completed boolean feature ({theme_name})")


** PROCESSING THEME: gender

	> Completed count feature (gender)
	> Completed boolean feature (gender)

** PROCESSING THEME: ethnicity

	> Completed count feature (ethnicity)
	> Completed boolean feature (ethnicity)

** PROCESSING THEME: sexuality

	> Completed count feature (sexuality)
	> Completed boolean feature (sexuality)

** PROCESSING THEME: ability

	> Completed count feature (ability)
	> Completed boolean feature (ability)

** PROCESSING THEME: age

	> Completed count feature (age)
	> Completed boolean feature (age)

** PROCESSING THEME: weight

	> Completed count feature (weight)
	> Completed boolean feature (weight)

** PROCESSING THEME: appearance

	> Completed count feature (appearance)
	> Completed boolean feature (appearance)

** PROCESSING THEME: class

	> Completed count feature (class)
	> Completed boolean feature (class)


In [44]:
## ADD SUMMARY THEME FEATURE
theme_names = list(THEMES.keys())

def themes_found_for_row(row):
    pairs = [(t, int(row[f"theme_{t}_count"])) for t in theme_names if row[f"theme_{t}_count"] > 0]
    pairs.sort(key=lambda x: x[1], reverse=True)  # highest count first
    return pairs  # list of (theme_name, count) tuples

merged_df["themes_found"] = merged_df.apply(themes_found_for_row, axis=1)

## REVIEW UPDATES
# show_df_details(["head", "tail", "info", "describe"])
# Review whole_text_normalised and all theme-related columns
# Show first and last several rows (jokes)
review_rows = pd.concat([
    merged_df[["whole_text_normalised", "themes_found"]].head(30),
    merged_df[["whole_text_normalised", "themes_found"]].tail(30)
])

for idx, row in review_rows.iterrows():
    print(f"\n@{idx}: {row["themes_found"]}:\n\t{row["whole_text_normalised"]}")


@0: [('ethnicity', 1)]:
	i hate how you cant even say black paint anymore now i have to say leroy can you please paint the fence

@1: [('ethnicity', 3)]:
	what s the difference between a jew in nazi germany and pizza pizza doesn t scream when you put it in the oven i m so sorry

@2: [('ethnicity', 3)]:
	i recently went to america and being there really helped me learn about american culture so i visited a shop and as i was leaving the shopkeeper said have a nice day but i didn t so i sued him

@3: [('gender', 1), ('class', 1)]:
	brian raises his hand and says he s in heaven a sunday school teacher is concerned that his students might be a little confused about jesus so he asks his class where is jesus today brian raises his hand and says he s in heaven susan answers he s in my heart little johnny waves his hand furiously and blurts out he s in our bathroom the teacher is surprised by this answer and asks little johnny how he knows this well little johnny says every morning my dad gets

##### Tidy dataframe - last steps

#### Output new CSV (in data folder)